# Conjunto de Datos 1: daily-total-female-births.csv


# 1. Análisis Exploratorio:

In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
import seaborn as sns
from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')


In [48]:
df = pd.read_csv('./daily-total-female-births.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
data = df['Births']

# División entrenamiento/prueba
train_size = int(len(data) * 0.8)
train, test = data[:train_size], data[train_size:]

print(f"Datos de entrenamiento: {len(train)}")
print(f"Datos de prueba: {len(test)}")

Datos de entrenamiento: 292
Datos de prueba: 73


In [49]:
from ydata_profiling import ProfileReport

# Generar reporte
profile = ProfileReport(
    df,
    title="Análisis de Nacimientos",
    explorative=True
)

# Guardar reporte
profile.to_file("./profiling_analisis/reporte1.html")

print("✅ Reporte generado: reporte_births.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 244.07it/s]

✅ Reporte generado: reporte_births.html


In [50]:
# Configuración de estilo
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Cargar datos
df = pd.read_csv('./daily-total-female-births.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

# Crear variables temporales adicionales
df['DayOfWeek'] = df.index.dayofweek
df['DayName'] = df.index.day_name()
df['Month'] = df.index.month
df['MonthName'] = df.index.month_name()
df['Quarter'] = df.index.quarter
df['DayOfYear'] = df.index.dayofyear
df['WeekOfYear'] = df.index.isocalendar().week

# 1. ESTADÍSTICAS DESCRIPTIVAS
print("=" * 60)
print("ANÁLISIS EXPLORATORIO DE NACIMIENTOS FEMENINOS DIARIOS - 1959")
print("=" * 60)

print("\n1. ESTADÍSTICAS DESCRIPTIVAS:")
print("-" * 40)
stats_summary = df['Births'].describe()
print(stats_summary)
print(f"\nAsimetría (Skewness): {df['Births'].skew():.3f}")
print(f"Curtosis: {df['Births'].kurtosis():.3f}")
print(f"Coeficiente de variación: {(df['Births'].std() / df['Births'].mean() * 100):.2f}%")

# 2. ANÁLISIS TEMPORAL
print("\n2. ANÁLISIS TEMPORAL:")
print("-" * 40)

# Estadísticas por día de la semana
day_stats = df.groupby('DayName')['Births'].agg(['mean', 'std', 'count'])
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_stats = day_stats.reindex(day_order)
print("\nNacimientos por día de la semana:")
print(day_stats.round(2))

# Estadísticas por mes
month_stats = df.groupby('MonthName')['Births'].agg(['mean', 'std', 'sum'])
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 
               'July', 'August', 'September', 'October', 'November', 'December']
month_stats = month_stats.reindex(month_order)
print("\nNacimientos por mes:")
print(month_stats.round(2))

# 3. ANÁLISIS DE OUTLIERS
print("\n3. ANÁLISIS DE OUTLIERS:")
print("-" * 40)
Q1 = df['Births'].quantile(0.25)
Q3 = df['Births'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['Births'] < lower_bound) | (df['Births'] > upper_bound)]
print(f"Límites IQR: [{lower_bound:.1f}, {upper_bound:.1f}]")
print(f"Número de outliers: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)")
if len(outliers) > 0:
    print("\nFechas con valores atípicos:")
    for idx, row in outliers.iterrows():
        print(f"  {idx.strftime('%Y-%m-%d')} ({idx.strftime('%A')}): {row['Births']} nacimientos")


ANÁLISIS EXPLORATORIO DE NACIMIENTOS FEMENINOS DIARIOS - 1959

1. ESTADÍSTICAS DESCRIPTIVAS:
----------------------------------------
count    365.000000
mean      41.980822
std        7.348257
min       23.000000
25%       37.000000
50%       42.000000
75%       46.000000
max       73.000000
Name: Births, dtype: float64

Asimetría (Skewness): 0.447
Curtosis: 0.778
Coeficiente de variación: 17.50%

2. ANÁLISIS TEMPORAL:
----------------------------------------

Nacimientos por día de la semana:
            mean   std  count
DayName                      
Monday     41.13  7.51     52
Tuesday    43.75  6.79     52
Wednesday  43.85  9.06     52
Thursday   43.08  6.40     53
Friday     41.96  6.28     52
Saturday   41.19  7.88     52
Sunday     38.88  6.19     52

Nacimientos por mes:
            mean   std   sum
MonthName                   
January    39.13  7.74  1213
February   41.00  7.94  1148
March      39.29  6.90  1218
April      39.83  7.21  1195
May        38.97  6.14  1208
June 

In [51]:
# Crear figura principal con diseño de dashboard
from matplotlib.gridspec import GridSpec
fig = plt.figure(figsize=(20, 12))
fig.patch.set_facecolor('#f8f9fa')

# Título principal del dashboard
fig.suptitle('DASHBOARD: ANÁLISIS DE NACIMIENTOS FEMENINOS DIARIOS - 1959', 
             fontsize=24, fontweight='bold', y=0.98)

# Crear una cuadrícula personalizada
gs = GridSpec(3, 3, figure=fig, height_ratios=[1, 1.2, 0.8], width_ratios=[1, 1, 1],
              hspace=0.3, wspace=0.25)

# ===========================
# 1. BOXPLOT POR DÍA DE LA SEMANA (Superior izquierda)
# ===========================
ax1 = fig.add_subplot(gs[0, :2])
ax1.set_facecolor('#ffffff')

# Preparar datos para boxplot
df_plot = df.copy()
df_plot['DayName'] = pd.Categorical(df_plot['DayName'], categories=day_order, ordered=True)

# Crear boxplot mejorado
bp = df_plot.boxplot(column='Births', by='DayName', ax=ax1, patch_artist=True,
                     boxprops=dict(facecolor='lightblue', alpha=0.8),
                     medianprops=dict(color='darkred', linewidth=2),
                     whiskerprops=dict(color='gray', linewidth=1.5),
                     capprops=dict(color='gray', linewidth=1.5),
                     flierprops=dict(marker='o', markerfacecolor='red', markersize=6, alpha=0.6))

# Personalizar
ax1.set_title('Distribución de Nacimientos por Día de la Semana', 
              fontsize=16, fontweight='bold', pad=15)
ax1.set_xlabel('Día de la Semana', fontsize=12, fontweight='bold')
ax1.set_ylabel('Número de Nacimientos', fontsize=12, fontweight='bold')
ax1.set_xticklabels(['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom'], rotation=0)
ax1.grid(True, alpha=0.3, axis='y')

# Añadir línea de media general
ax1.axhline(y=df['Births'].mean(), color='green', linestyle='--', 
            linewidth=2, alpha=0.7, label=f'Media general: {df["Births"].mean():.1f}')
ax1.legend(loc='upper right')

# Eliminar título automático del boxplot
fig.texts = [text for text in fig.texts if 'Boxplot grouped by' not in text.get_text()]

# ===========================
# 2. COMPARACIÓN FIN DE SEMANA VS DÍAS LABORABLES (Superior derecha)
# ===========================
ax2 = fig.add_subplot(gs[0, 2])
ax2.set_facecolor('#ffffff')

# Calcular comparación
df['IsWeekend'] = df['DayOfWeek'].isin([5, 6])
weekend_comparison = df.groupby('IsWeekend')['Births'].agg(['mean', 'std', 'count'])
weekend_comparison.index = ['Días laborables', 'Fin de semana']

# Crear gráfico de barras con barras de error
bars = ax2.bar(weekend_comparison.index, weekend_comparison['mean'], 
                yerr=weekend_comparison['std'], capsize=10,
                color=['#3498db', '#e74c3c'], alpha=0.8, edgecolor='black', linewidth=1.5)

# Añadir valores encima de las barras
for i, (idx, row) in enumerate(weekend_comparison.iterrows()):
    ax2.text(i, row['mean'] + row['std'] + 0.5, f"{row['mean']:.1f}", 
             ha='center', va='bottom', fontsize=14, fontweight='bold')
    # Añadir diferencia porcentual
    if i == 1:
        diff_pct = ((weekend_comparison.iloc[0]['mean'] - row['mean']) / row['mean'] * 100)
        ax2.text(0.5, row['mean'] + 5, f'+{diff_pct:.1f}%', 
                 ha='center', fontsize=12, color='green', fontweight='bold')

ax2.set_title('Promedio de Nacimientos:\nDías Laborables vs Fin de Semana', 
              fontsize=14, fontweight='bold', pad=15)
ax2.set_ylabel('Promedio de Nacimientos', fontsize=12, fontweight='bold')
ax2.set_ylim(0, weekend_comparison['mean'].max() * 1.3)
ax2.grid(True, alpha=0.3, axis='y')

# ===========================
# 3. SERIE TEMPORAL CON MEDIAS MÓVILES (Centro)
# ===========================
ax3 = fig.add_subplot(gs[1, :])
ax3.set_facecolor('#ffffff')

# Calcular medias móviles
df['MA7'] = df['Births'].rolling(window=7).mean()
df['MA30'] = df['Births'].rolling(window=30).mean()

# Graficar serie temporal
df['Births'].plot(ax=ax3, alpha=0.4, label='Datos diarios', color='lightgray', linewidth=0.8)
df['MA7'].plot(ax=ax3, label='Media móvil 7 días', linewidth=2.5, color='#2ecc71')
df['MA30'].plot(ax=ax3, label='Media móvil 30 días', linewidth=2.5, color='#e74c3c')

# Resaltar máximo y mínimo
max_idx = df['Births'].idxmax()
min_idx = df['Births'].idxmin()
ax3.scatter(max_idx, df.loc[max_idx, 'Births'], color='red', s=150, zorder=5, 
           label=f'Máximo: {df.loc[max_idx, "Births"]}')
ax3.scatter(min_idx, df.loc[min_idx, 'Births'], color='darkred', s=150, zorder=5,
           label=f'Mínimo: {df.loc[min_idx, "Births"]}')

# Añadir anotaciones
ax3.annotate(f'{df.loc[max_idx, "Births"]} nacimientos\n{max_idx.strftime("%d %b")}', 
             xy=(max_idx, df.loc[max_idx, 'Births']), 
             xytext=(10, 20), textcoords='offset points',
             bbox=dict(boxstyle="round,pad=0.3", facecolor='yellow', alpha=0.7),
             arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))

ax3.set_title('Serie Temporal de Nacimientos con Medias Móviles', 
              fontsize=18, fontweight='bold', pad=15)
ax3.set_xlabel('Fecha', fontsize=12, fontweight='bold')
ax3.set_ylabel('Número de Nacimientos', fontsize=12, fontweight='bold')
ax3.legend(loc='upper left', fontsize=11, framealpha=0.9)
ax3.grid(True, alpha=0.3)

# Sombrear fines de semana
for i in range(len(df)):
    if df.index[i].dayofweek >= 5:  # Sábado o domingo
        ax3.axvspan(df.index[i], df.index[i] + pd.Timedelta(days=1), 
                   alpha=0.1, color='orange')

# ===========================
# 4. TOTAL DE NACIMIENTOS POR MES (Inferior)
# ===========================
ax4 = fig.add_subplot(gs[2, :])
ax4.set_facecolor('#ffffff')

# Calcular totales mensuales
monthly_births = df.groupby('Month')['Births'].agg(['sum', 'mean', 'std'])
month_names = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 
               'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

# Crear gráfico de barras
bars = ax4.bar(range(1, 13), monthly_births['sum'], color='coral', alpha=0.8, 
                edgecolor='black', linewidth=1.5)

# Colorear el mes con más y menos nacimientos
max_month = monthly_births['sum'].idxmax()
min_month = monthly_births['sum'].idxmin()
bars[max_month-1].set_color('#27ae60')
bars[min_month-1].set_color('#c0392b')

# Añadir línea de promedio
ax4.axhline(y=monthly_births['sum'].mean(), color='blue', linestyle='--', 
            linewidth=2, alpha=0.7, label=f'Promedio mensual: {monthly_births["sum"].mean():.0f}')

# Añadir valores encima de las barras
for i, (idx, row) in enumerate(monthly_births.iterrows()):
    ax4.text(i+1, row['sum'] + 10, f"{row['sum']:,}", 
             ha='center', va='bottom', fontsize=10, fontweight='bold')

ax4.set_title('Total de Nacimientos por Mes', fontsize=16, fontweight='bold', pad=15)
ax4.set_xlabel('Mes', fontsize=12, fontweight='bold')
ax4.set_ylabel('Total de Nacimientos', fontsize=12, fontweight='bold')
ax4.set_xticks(range(1, 13))
ax4.set_xticklabels(month_names, rotation=0)
ax4.legend(loc='upper right')
ax4.grid(True, alpha=0.3, axis='y')

# ===========================
# PANEL DE ESTADÍSTICAS RESUMEN
# ===========================
# Crear un texto con estadísticas clave
stats_text = f"""
ESTADÍSTICAS CLAVE:
• Total de nacimientos en 1959: {df['Births'].sum():,}
• Promedio diario: {df['Births'].mean():.1f} ± {df['Births'].std():.1f}
• Diferencia laborables vs fin de semana: {weekend_comparison.iloc[0]['mean'] - weekend_comparison.iloc[1]['mean']:.1f} nacimientos/día
• Mes con más nacimientos: {month_names[max_month-1]} ({monthly_births.loc[max_month, 'sum']:,})
• Mes con menos nacimientos: {month_names[min_month-1]} ({monthly_births.loc[min_month, 'sum']:,})
"""

# Añadir el texto al dashboard
fig.text(0.02, 0.02, stats_text, fontsize=12, 
         bbox=dict(boxstyle="round,pad=0.5", facecolor='lightgray', alpha=0.8),
         verticalalignment='bottom')

# Ajustar el layout

# Guardar el dashboard
plt.savefig('./imagenes/dashboard_nacimientos.png', dpi=300, bbox_inches='tight', facecolor='#f8f9fa')
plt.tight_layout()
plt.show()

# 2. Promedios Móviles:

In [52]:
# Calcular promedios móviles
ma_7 = train.rolling(window=7).mean()
ma_14 = train.rolling(window=14).mean() 
ma_30 = train.rolling(window=30).mean()

# Extender predicciones para el conjunto de prueba
ma_7_extended = data.rolling(window=7).mean()[train_size:]
ma_14_extended = data.rolling(window=14).mean()[train_size:]
ma_30_extended = data.rolling(window=30).mean()[train_size:]

# Calcular RMSE
rmse_ma7 = np.sqrt(mean_squared_error(test, ma_7_extended))
rmse_ma14 = np.sqrt(mean_squared_error(test, ma_14_extended))
rmse_ma30 = np.sqrt(mean_squared_error(test, ma_30_extended))

print("RMSE Promedios Móviles:")
print(f"MA_7: {rmse_ma7:.2f}")
print(f"MA_14: {rmse_ma14:.2f}")
print(f"MA_30: {rmse_ma30:.2f}")

RMSE Promedios Móviles:
MA_7: 5.80
MA_14: 6.78
MA_30: 6.66


In [53]:
# Gráfica Promedios Móviles
plt.figure(figsize=(12, 6))
plt.plot(data.index, data.values, label='Original', alpha=0.7, color='gray')
plt.plot(ma_7.index, ma_7.values, label=f'MA_7 (RMSE: {rmse_ma7:.2f})', linewidth=2)
plt.plot(ma_14.index, ma_14.values, label=f'MA_14 (RMSE: {rmse_ma14:.2f})', linewidth=2)
plt.plot(ma_30.index, ma_30.values, label=f'MA_30 (RMSE: {rmse_ma30:.2f})', linewidth=2)
plt.axvline(x=data.index[train_size], color='red', linestyle='--', label='División Train/Test')
plt.title('Promedios Móviles')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('./imagenes/BIRTHSpromediomoviles.png',facecolor='#f8f9fa')
plt.show()

# 3. Alisamiento Exponencial:

In [54]:
# Alisamiento Exponencial Simple
model_simple = ExponentialSmoothing(train, trend=None, seasonal=None)
fit_simple = model_simple.fit()
forecast_simple = fit_simple.forecast(len(test))
rmse_simple = np.sqrt(mean_squared_error(test, forecast_simple))

# Alisamiento Exponencial Doble (Holt)
model_double = ExponentialSmoothing(train, trend='add', seasonal=None)
fit_double = model_double.fit()
forecast_double = fit_double.forecast(len(test))
rmse_double = np.sqrt(mean_squared_error(test, forecast_double))

print("RMSE Alisamiento Exponencial:")
print(f"Simple: {rmse_simple:.2f}")
print(f"Doble (Holt): {rmse_double:.2f}")

RMSE Alisamiento Exponencial:
Simple: 7.54
Doble (Holt): 8.47


In [55]:
# Gráfica Alisamiento Exponencial
plt.figure(figsize=(12, 6))
plt.plot(train.index, train.values, label='Train', color='blue')
plt.plot(test.index, test.values, label='Test', color='green')
plt.plot(test.index, forecast_simple, label=f'Simple (RMSE: {rmse_simple:.2f})', linewidth=2)
plt.plot(test.index, forecast_double, label=f'Doble/Holt (RMSE: {rmse_double:.2f})', linewidth=2)
plt.axvline(x=data.index[train_size], color='red', linestyle='--', label='División Train/Test')
plt.title('Alisamiento Exponencial')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('./imagenes/BIRTHSalisamientoexponencial.png',facecolor='#f8f9fa')
plt.show()

# 4. HOLT-WINTERS

In [56]:
# Holt-Winters Aditivo
model_hw_add = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=7)
fit_hw_add = model_hw_add.fit()
forecast_hw_add = fit_hw_add.forecast(len(test))
rmse_hw_add = np.sqrt(mean_squared_error(test, forecast_hw_add))

# Holt-Winters Multiplicativo
try:
    model_hw_mult = ExponentialSmoothing(train, trend='add', seasonal='mul', seasonal_periods=7)
    fit_hw_mult = model_hw_mult.fit()
    forecast_hw_mult = fit_hw_mult.forecast(len(test))
    rmse_hw_mult = np.sqrt(mean_squared_error(test, forecast_hw_mult))
    mult_success = True
except:
    mult_success = False
    print("Modelo multiplicativo falló")

print("RMSE Holt-Winters:")
print(f"Aditivo: {rmse_hw_add:.2f}")
if mult_success:
    print(f"Multiplicativo: {rmse_hw_mult:.2f}")

RMSE Holt-Winters:
Aditivo: 8.89
Multiplicativo: 11.35


In [57]:

# Gráfica Holt-Winters
plt.figure(figsize=(12, 6))
plt.plot(train.index, train.values, label='Train', color='blue')
plt.plot(test.index, test.values, label='Test', color='green')
plt.plot(test.index, forecast_hw_add, label=f'HW Aditivo (RMSE: {rmse_hw_add:.2f})', linewidth=2)
if mult_success:
    plt.plot(test.index, forecast_hw_mult, label=f'HW Multiplicativo (RMSE: {rmse_hw_mult:.2f})', linewidth=2)
plt.axvline(x=data.index[train_size], color='red', linestyle='--', label='División Train/Test')
plt.title('Holt-Winters')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('./imagenes/BIRTHSHoltWinterpng',facecolor='#f8f9fa')
plt.show()

# 5. SARIMA:

In [58]:
# Búsqueda de parámetros SARIMA
print("Buscando parámetros SARIMA...")
best_aic = float('inf')
best_params = None

for p in range(3):
    for d in range(2):
        for q in range(3):
            for P in range(2):
                for D in range(2):
                    for Q in range(2):
                        try:
                            model = SARIMAX(train, order=(p,d,q), seasonal_order=(P,D,Q,7))
                            fit = model.fit(disp=False)
                            if fit.aic < best_aic:
                                best_aic = fit.aic
                                best_params = ((p,d,q), (P,D,Q,7))
                        except:
                            continue

# Ajustar mejor modelo SARIMA
if best_params:
    order, seasonal_order = best_params
    sarima_model = SARIMAX(train, order=order, seasonal_order=seasonal_order)
    sarima_fit = sarima_model.fit(disp=False)
    sarima_forecast = sarima_fit.forecast(len(test))
    rmse_sarima = np.sqrt(mean_squared_error(test, sarima_forecast))
    
    # Gráfica SARIMA
    plt.figure(figsize=(12, 6))
    plt.plot(train.index, train.values, label='Train', color='blue')
    plt.plot(test.index, test.values, label='Test', color='green')
    plt.plot(test.index, sarima_forecast, label=f'SARIMA{order}x{seasonal_order} (RMSE: {rmse_sarima:.2f})', linewidth=2)
    plt.axvline(x=data.index[train_size], color='red', linestyle='--', label='División Train/Test')
    plt.title('SARIMA')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('./imagenes/BIRTHSSarima.png',facecolor='#f8f9fa')
    plt.show()
    
    print(f"Mejores parámetros SARIMA: {best_params}")
    print(f"RMSE SARIMA: {rmse_sarima:.2f}")


Buscando parámetros SARIMA...
Mejores parámetros SARIMA: ((0, 1, 1), (0, 1, 1, 7))
RMSE SARIMA: 9.74


# 6. Prophet:

In [59]:
try:
    from prophet import Prophet
    
    # Preparar datos para Prophet
    prophet_data = data.reset_index()
    prophet_data.columns = ['ds', 'y']
    prophet_train = prophet_data[:train_size]
    
    # Crear y entrenar modelo Prophet
    prophet_model = Prophet(daily_seasonality=True, yearly_seasonality=False)
    prophet_model.fit(prophet_train)
    
    # Hacer predicciones
    future = prophet_model.make_future_dataframe(periods=len(test))
    prophet_forecast = prophet_model.predict(future)
    prophet_pred = prophet_forecast['yhat'][train_size:].values
    rmse_prophet = np.sqrt(mean_squared_error(test, prophet_pred))
    
    # Gráfica Prophet
    plt.figure(figsize=(12, 6))
    plt.plot(train.index, train.values, label='Train', color='blue')
    plt.plot(test.index, test.values, label='Test', color='green')
    plt.plot(test.index, prophet_pred, label=f'Prophet (RMSE: {rmse_prophet:.2f})', linewidth=2)
    plt.axvline(x=data.index[train_size], color='red', linestyle='--', label='División Train/Test')
    plt.title('Prophet')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('./imagenes/BIRTHSprophet.png',facecolor='#f8f9fa')
    plt.show()
    
    print(f"RMSE Prophet: {rmse_prophet:.2f}")
    
except ImportError:
    print("Prophet no está instalado. Instalar con: pip install prophet")


12:27:49 - cmdstanpy - INFO - Chain [1] start processing
12:27:50 - cmdstanpy - INFO - Chain [1] done processing


RMSE Prophet: 9.60


# 7. Comparación y Evaluación:


In [60]:
# Recopilar todos los RMSE
all_rmse = {
    'MA_7': rmse_ma7,
    'MA_14': rmse_ma14,
    'MA_30': rmse_ma30,
    'Exp_Simple': rmse_simple,
    'Exp_Doble': rmse_double,
    'HW_Aditivo': rmse_hw_add
}

if mult_success:
    all_rmse['HW_Multiplicativo'] = rmse_hw_mult

if 'rmse_sarima' in locals():
    all_rmse['SARIMA'] = rmse_sarima

if 'rmse_prophet' in locals():
    all_rmse['Prophet'] = rmse_prophet

# Gráfica comparación de errores
plt.figure(figsize=(12, 6))
models = list(all_rmse.keys())
rmse_values = list(all_rmse.values())
bars = plt.bar(models, rmse_values, color='skyblue', edgecolor='black')

# Resaltar el mejor modelo
min_idx = rmse_values.index(min(rmse_values))
bars[min_idx].set_color('green')

plt.title('Comparación de RMSE por Modelo')
plt.ylabel('RMSE')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')

# Añadir valores encima de las barras
for i, v in enumerate(rmse_values):
    plt.text(i, v + 0.1, f'{v:.2f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()
plt.savefig("./imagenes/BIRTHScomparacion.png")


print("COMPARACIÓN FINAL DE RMSE:")
for model, rmse in sorted(all_rmse.items(), key=lambda x: x[1]):
    print(f"{model}: {rmse:.2f}")

COMPARACIÓN FINAL DE RMSE:
MA_7: 5.80
MA_30: 6.66
MA_14: 6.78
Exp_Simple: 7.54
Exp_Doble: 8.47
HW_Aditivo: 8.89
Prophet: 9.60
SARIMA: 9.74
HW_Multiplicativo: 11.35
